# Evaluating the Advanced LangGraph Product Agent with LangSmith

This notebook evaluates the **second LangGraph product-description workflow** from the classroom notebook.

The workflow uses:

- A basic-description node
- Three parallel nodes for features, audience, and SEO
- A marketing merge node
- A final polishing node

LangSmith evaluation has three main parts:

1. **Dataset** – test examples given to the agent
2. **Target function** – the application or agent being tested
3. **Evaluators** – functions that score the output

We will use four simple metrics:

| Metric | What it checks |
|---|---|
| `non_empty` | Did the agent generate an output? |
| `product_name_present` | Does the output mention the product? |
| `keyword_coverage` | How many expected keywords appear? |
| `length_score` | Is the description within a useful word range? |

These are **code-based evaluators**, so they are easy to explain, repeatable, and do not require an additional evaluator LLM.


In [1]:
!pip install -qU langgraph langchain-openai langsmith typing-extensions


## 1. Configure API Keys and LangSmith Tracing

`LANGSMITH_TRACING=true` sends the LangGraph and LLM execution traces to LangSmith.

`LANGSMITH_PROJECT` groups the traces under one project name.

The OpenRouter key is used by the product-description agent.


In [2]:
import os
import getpass

os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OpenRouter API key: ")
os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter LangSmith API key: ")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langgraph-product-agent-evaluation"

print("LangSmith tracing configured.")


LangSmith tracing configured.


## 2. Imports and LLM Setup

The same OpenRouter-based `ChatOpenAI` setup is used as in the original agent notebook.


In [3]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langsmith import Client

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.3,
    openai_api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)


## 3. Define the Advanced Agent State

The state stores the input and every intermediate result generated by the graph.

Each node returns only the state fields that it updates.


In [4]:
class State(TypedDict):
    product_name: str
    basic_description: str
    features_benefits: str
    target_audience: str
    seo_keywords: str
    marketing_message: str
    final_description: str


## 4. Define the Agent Nodes

After the basic description is created, the `features`, `audience`, and `seo` nodes can run independently.

Their outputs are then combined by the `marketing` node.


In [5]:
def generate_basic_description(state: State):
    response = llm.invoke([
        SystemMessage(content="You generate short and clear product descriptions."),
        HumanMessage(content=f"Write a brief description of '{state['product_name']}'."),
    ])
    return {"basic_description": response.content}


def add_features(state: State):
    response = llm.invoke(
        f"List the key features and customer benefits of this product:"
        f"{state['basic_description']}"
    )
    return {"features_benefits": response.content}


def identify_audience(state: State):
    response = llm.invoke(
        f"Identify the ideal target audience for this product:"
        f"{state['basic_description']}"
    )
    return {"target_audience": response.content}


def generate_seo_keywords(state: State):
    response = llm.invoke(
        f"Generate relevant SEO keywords for this product:"
        f"{state['basic_description']}"
    )
    return {"seo_keywords": response.content}


def create_marketing_message(state: State):
    combined_information = f"""
Features and benefits:
{state['features_benefits']}

Target audience:
{state['target_audience']}

SEO keywords:
{state['seo_keywords']}
"""

    response = llm.invoke(
        "Create a persuasive marketing message using the information below. "
        "Keep the claims realistic."
        + combined_information
    )
    return {"marketing_message": response.content}


def polish_final_description(state: State):
    response = llm.invoke(
        f"""
Polish the marketing message into one final product description.
Mention the product name, explain its value clearly, and keep it between 40 and 140 words.

Product name: {state['product_name']}

Marketing message:
{state['marketing_message']}
"""
    )
    return {"final_description": response.content}


## 5. Build the LangGraph Workflow

The graph uses **fan-out** after `basic` and **fan-in** before `marketing`.

```text
                         ┌→ features ─┐
START → basic description├→ audience ─┼→ marketing → final → END
                         └→ SEO ──────┘
```

The random internal quality score from the classroom demonstration is omitted here. LangSmith now performs the external evaluation in a repeatable way.


In [6]:
def build_workflow():
    workflow = StateGraph(State)

    workflow.add_node("basic", generate_basic_description)
    workflow.add_node("features", add_features)
    workflow.add_node("audience", identify_audience)
    workflow.add_node("seo", generate_seo_keywords)
    workflow.add_node("marketing", create_marketing_message)
    workflow.add_node("final", polish_final_description)

    workflow.add_edge(START, "basic")

    # Fan-out
    workflow.add_edge("basic", "features")
    workflow.add_edge("basic", "audience")
    workflow.add_edge("basic", "seo")

    # Fan-in
    workflow.add_edge("features", "marketing")
    workflow.add_edge("audience", "marketing")
    workflow.add_edge("seo", "marketing")

    workflow.add_edge("marketing", "final")
    workflow.add_edge("final", END)

    return workflow.compile()


app = build_workflow()
print("Workflow compiled successfully.")


Workflow compiled successfully.


## 6. Test the Agent Once

Before running a complete evaluation, test one input and inspect the final state.


In [7]:
def run_agent(product_name: str) -> dict:
    initial_state: State = {
        "product_name": product_name,
        "basic_description": "",
        "features_benefits": "",
        "target_audience": "",
        "seo_keywords": "",
        "marketing_message": "",
        "final_description": "",
    }
    return app.invoke(initial_state)


test_result = run_agent("Smart Water Bottle")
print(test_result["final_description"])


**Smart Water Bottle: Your Ultimate Hydration Companion**

Elevate your hydration game with the **Smart Water Bottle**—the perfect blend of technology and style for health-conscious individuals. This innovative water bottle features water intake tracking, personalized hydration goals, and timely reminders to keep you on track throughout your day. Its sleek, BPA-free design makes it a fashionable accessory for the gym, office, or outdoor adventures. With a built-in LED indicator and smartphone syncing capabilities, you can effortlessly monitor your hydration levels and stay motivated. Lightweight and portable, the Smart Water Bottle is your go-to solution for a healthier lifestyle. Make hydration a priority—order your Smart Water Bottle today and transform the way you drink!


## 7. Create a Small LangSmith Dataset

Each dataset example contains:

- `inputs`: the product name sent to the graph
- `outputs`: reference information used by the evaluators

The reference output is not a complete ideal description. It only contains a few expected keywords.


In [8]:
client = Client()
DATASET_NAME = "Advanced Product Agent - Simple Evaluation"

examples = [
    {
        "inputs": {"product_name": "Smart Water Bottle"},
        "outputs": {"expected_keywords": ["water", "smart", "hydration"]},
    },
    {
        "inputs": {"product_name": "Noise Cancelling Headphones"},
        "outputs": {"expected_keywords": ["noise", "sound", "comfort"]},
    },
    {
        "inputs": {"product_name": "Portable Solar Charger"},
        "outputs": {"expected_keywords": ["solar", "portable", "charge"]},
    },
    {
        "inputs": {"product_name": "Ergonomic Office Chair"},
        "outputs": {"expected_keywords": ["ergonomic", "comfort", "support"]},
    },
]

# Reuse the dataset when the cell is executed again.
existing_datasets = list(client.list_datasets(dataset_name=DATASET_NAME))

if existing_datasets:
    dataset = existing_datasets[0]
    print(f"Using existing dataset: {dataset.name}")
else:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Simple evaluation dataset for the advanced LangGraph product agent.",
    )
    client.create_examples(dataset_id=dataset.id, examples=examples)
    print(f"Created dataset: {dataset.name}")


Created dataset: Advanced Product Agent - Simple Evaluation


## 8. Define the LangSmith Target Function

LangSmith sends each dataset input dictionary to this function.

The target runs the complete LangGraph workflow and returns the fields we want to inspect and evaluate.


In [9]:
def target(inputs: dict) -> dict:
    result = run_agent(inputs["product_name"])

    return {
        "final_description": result["final_description"],
        "basic_description": result["basic_description"],
        "features_benefits": result["features_benefits"],
        "target_audience": result["target_audience"],
        "seo_keywords": result["seo_keywords"],
        "marketing_message": result["marketing_message"],
    }


## 9. Define Four Simple Evaluators

An evaluator receives:

- `inputs`: the dataset input
- `outputs`: the actual agent output
- `reference_outputs`: the expected information stored in the dataset

A Boolean result becomes `1` for success and `0` for failure. A float can represent a partial score from `0` to `1`.


In [10]:
def non_empty(inputs: dict, outputs: dict) -> bool:
    """Checks whether a final description was generated."""
    description = outputs.get("final_description", "").strip()
    return bool(description)


def product_name_present(inputs: dict, outputs: dict) -> bool:
    """Checks whether the product name appears in the final description."""
    product_name = inputs["product_name"].lower()
    description = outputs.get("final_description", "").lower()
    return product_name in description


def keyword_coverage(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> float:
    """Returns the proportion of expected keywords found in the output."""
    expected = reference_outputs.get("expected_keywords", [])
    description = outputs.get("final_description", "").lower()

    if not expected:
        return 1.0

    matches = sum(keyword.lower() in description for keyword in expected)
    return matches / len(expected)


def length_score(inputs: dict, outputs: dict) -> float:
    """Gives full score for 40-140 words and partial credit outside the range."""
    description = outputs.get("final_description", "")
    word_count = len(description.split())

    if 40 <= word_count <= 140:
        return 1.0
    if 25 <= word_count < 40 or 140 < word_count <= 180:
        return 0.5
    return 0.0


## 10. Run the LangSmith Evaluation

`client.evaluate()` performs the following steps:

1. Reads every example from the dataset
2. Sends the input to `target()`
3. Runs the four evaluators
4. Saves the traces, outputs, and scores as a LangSmith experiment



In [11]:
experiment_results = client.evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[
        non_empty,
        product_name_present,
        keyword_coverage,
        length_score,
    ],
)

print(experiment_results)

View the evaluation results for experiment: 'enchanted-health-8' at:
https://smith.langchain.com/o/2eb63d0c-5d76-422a-ad6a-65decf2adcd7/datasets/5ca7c3af-cd21-48a8-8561-03a385903cf3/compare?selectedSessions=255a6075-c7c6-49cb-b979-0767715f5150




0it [00:00, ?it/s]

<ExperimentResults enchanted-health-8>


## 11. How to Read the Results in LangSmith

Open the experiment link printed by the previous cell.

For each test example, LangSmith shows:

- The product input
- The generated final description
- The complete execution trace
- The score for each evaluator

### Interpreting the metrics

- `non_empty = 1`: the workflow returned text
- `product_name_present = 1`: the product name appears in the final copy
- `keyword_coverage = 0.67`: two of three expected keywords appeared
- `length_score = 1`: the output had between 40 and 140 words

These metrics do not prove that the writing is perfect. They provide simple, consistent signals that can help compare prompt or model versions.


## 12. Optional Classroom Experiment

Change one part of the agent, run the evaluation again, and compare experiments in LangSmith.

Examples:

- Change the LLM temperature
- Shorten the final prompt
- Remove SEO keywords from the marketing node
- Use another model
- Change the desired word range

A good evaluation workflow helps answer:

> Did the new version perform better, worse, or approximately the same on the same test cases?
